<a href="https://colab.research.google.com/github/cursuri-inf/IPAP/blob/main/saptamana-02/L2_primul_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Caietul „Primul model” (Laborator 2)

Antrenezi primul tău clasificator de imagini: pornești de la o rețea antrenată deja pe peste un milion de fotografii și o adaptezi, cu 244 de imagini, ca să deosebească furnicile de albine.

Pașii:

1. Deschizi caietul cu butonul „Open in Colab” de mai sus. La prima rulare, Colab avertizează că documentul nu a fost creat de Google: caietul vine din depozitul cursului, așa că alegi „Run anyway”.
2. Înainte să scrii ceva: File → Save a copy in Drive; lucrezi în copie.
3. Lucrezi în pereche cu un coleg din echipă. Pilotul rulează și scrie, navigatorul citește, prezice și verifică; schimbați rolurile la fiecare secțiune. Într-o echipă de 3, al treilea coleg este verificator: rulează în copia lui și compară rezultatele. Fiecare predă caietul lui.
4. Rulezi celulele în ordine, cu Shift+Enter. La fiecare „Exercițiul B…” te oprești și scrii răspunsul în celula „Răspunsul tău:” de sub el.
5. La final: Runtime → Restart session and run all (rularea de la zero), verifici că ultima celulă de cod afișează „Caietul a rulat până la capăt”, completezi „Declarația AI”, descarci caietul (File → Download → Download .ipynb) și îl încarci la activitatea L2 din Campus Virtual.

Nivelul de bază (secțiunile 0–6 și exercițiile B1–B4) rulează pe procesor (CPU); nu ai nevoie de GPU. Extensiile E1–E4 sunt opționale, pentru nota 8–10.

Completează:

- Numele tău: …
- Colegul de pereche: …
- Echipa: …

## Harta caietului

Caietul urmează pașii fluxului de la curs, de la date la evaluare: date → model → eroare → antrenare → evaluare.

| Secțiunea | Pasul din flux |
|---|---|
| 1. Datele | date |
| 2. Referința simplă | pregătirea evaluării: cu ce comparăm |
| 3. Modelul preantrenat, fără antrenare | model |
| 4. Învățarea prin transfer | eroare și antrenare |
| 5. Evaluarea | evaluare |
| 6. Modelul pe o imagine nouă | folosirea modelului |

## 0. Pornire

Verifici versiunile, alegi dispozitivul (GPU dacă există, altfel procesorul) și fixezi semințele generatorului de numere aleatoare, ca rezultatele să fie aceleași la fiecare rulare. Pe GPU, cifrele pot diferi puțin de cele de pe procesor.

In [ ]:
import io, os, sys, time, random, shutil, zipfile, urllib.request
from collections import Counter

import numpy as np
import torch
from torch import nn
import torchvision
from torchvision import datasets, transforms, models
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

T_START = time.time()  # pentru durata totală, afișată la final

print("Python     ", sys.version.split()[0])
print("PyTorch    ", torch.__version__)
print("torchvision", torchvision.__version__)

DISPOZITIV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DISPOZITIV.type == "cuda":
    print("Dispozitiv  GPU,", torch.cuda.get_device_name(0))
else:
    print("Dispozitiv  CPU (procesorul; ajunge pentru nivelul de bază)")

SAMANTA = 42  # aceeași sămânță, aceleași rezultate
random.seed(SAMANTA)
np.random.seed(SAMANTA)
torch.manual_seed(SAMANTA)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

NR_FIRE = min(2, os.cpu_count() or 1)  # câte procese citesc imaginile în paralel


def procent(x):
    """0.9542 -> '95,4%' (cu virgulă zecimală)."""
    return f"{100 * x:.1f}%".replace(".", ",")


# culorile graficelor cursului
TEXT, SECUNDAR, ACCENT, SERIE2 = "#161616", "#4A4A4A", "#9B1B30", "#4E6375"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": False,
                     "text.color": TEXT, "axes.labelcolor": SECUNDAR,
                     "xtick.color": SECUNDAR, "ytick.color": SECUNDAR})

## 1. Datele

Pasul din flux: date.

Setul „hymenoptera” vine din tutorialul oficial PyTorch: fotografii de furnici (ants) și de albine (bees), împărțite deja în antrenare (train) și validare (val). Modelul învață doar din imaginile de antrenare; pe cele de validare nu le vede la antrenare, ca să putem măsura cât de bine se descurcă pe imagini noi. Aici validarea ține și loc de test; la proiect vei avea trei împărțiri (antrenare, validare, test), ca la curs.

In [ ]:
URL_DATE = "https://download.pytorch.org/tutorial/hymenoptera_data.zip"
ARHIVA = "hymenoptera_data.zip"
DATE = "hymenoptera_data"


def descarca(url, fisier, incercari=3):
    """Descarcă un fișier; dacă transferul se întrerupe, mai încearcă."""
    for incercare in range(1, incercari + 1):
        try:
            urllib.request.urlretrieve(url, fisier + ".part")
            os.replace(fisier + ".part", fisier)   # fișierul apare doar când e complet
            return
        except Exception as e:
            print(f"Încercarea {incercare} nu a reușit: {e}")
    raise RuntimeError("Descărcarea nu a reușit. Verifică conexiunea și rulează din nou celula.")


if not os.path.isdir(DATE):  # descărcăm doar dacă datele nu există deja
    if not (os.path.exists(ARHIVA) and zipfile.is_zipfile(ARHIVA)):   # lipsește sau e incompletă
        print("Descarc datele...")
        descarca(URL_DATE, ARHIVA)
    with zipfile.ZipFile(ARHIVA) as z:
        z.extractall(".")
print("Datele sunt în", os.path.abspath(DATE))

In [ ]:
# câte imagini are fiecare clasă, în fiecare împărțire
for impartire in ["train", "val"]:
    for clasa in sorted(os.listdir(os.path.join(DATE, impartire))):
        fisiere = os.listdir(os.path.join(DATE, impartire, clasa))
        imagini = [f for f in fisiere if f.lower().endswith((".jpg", ".jpeg"))]
        alte = [f for f in fisiere if f not in imagini]
        print(f"{impartire:5} {clasa}: {len(imagini):3} imagini", f"(și alte fișiere: {alte})" if alte else "")

Observă fișierul `imageNotFound.gif` din folderul furnicilor: o „imagine” care nu arată nicio furnică, rămasă din greșeală în set. Datele reale au astfel de defecte. Aici nu ne încurcă, pentru că `ImageFolder`, folosit mai jos, citește doar formatele obișnuite de imagine și ignoră fișierele .gif.

### Pregătirea imaginilor

Rețeaua pe care o folosim a fost antrenată pe imagini de 224 × 224 pixeli, cu culorile normalizate după media și abaterea standard ale setului ImageNet. Pregătim imaginile noastre la fel: le micșorăm (latura mică la 256 de pixeli), tăiem centrul de 224 × 224, le transformăm în tensori (tablouri de numere) și le normalizăm. Imaginile se citesc în loturi (batch) de câte 16.

In [ ]:
MEDIE, ABATERE = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]  # ale setului ImageNet
pregatire = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),                 # imagine -> tensor 3 x 224 x 224, valori între 0 și 1
    transforms.Normalize(MEDIE, ABATERE),
])

date_antrenare = datasets.ImageFolder(os.path.join(DATE, "train"), pregatire)
date_validare = datasets.ImageFolder(os.path.join(DATE, "val"), pregatire)
CLASE = date_antrenare.classes
print("Clasele:", CLASE, "(eticheta 0, eticheta 1)")
print("Imagini de antrenare:", len(date_antrenare), "| de validare:", len(date_validare))

MARIME_LOT = 16
incarcator_antrenare = torch.utils.data.DataLoader(
    date_antrenare, batch_size=MARIME_LOT, shuffle=True, num_workers=NR_FIRE,
    generator=torch.Generator().manual_seed(SAMANTA))   # ordinea amestecată, dar reproductibilă
incarcator_validare = torch.utils.data.DataLoader(
    date_validare, batch_size=MARIME_LOT, shuffle=False, num_workers=NR_FIRE)

### Exercițiul B1 (prezice ce afișează)

Celula de cod de după răspuns ia primul lot din `incarcator_antrenare` și afișează `imagini.shape` și `etichete`. Înainte s-o rulezi, scrie în celula de mai jos ce crezi că va afișa: câte numere are forma lotului și ce înseamnă fiecare? Ce valori pot avea etichetele? Indiciu: uită-te la `MARIME_LOT`, la cele trei canale de culoare (roșu, verde, albastru) și la mărimea tăieturii din `pregatire`.

Răspunsul tău: (predicția, scrisă înainte de rulare; după rulare, o frază: ai avut dreptate?)

In [ ]:
imagini, etichete = next(iter(incarcator_antrenare))
print("imagini.shape:", imagini.shape)
print("etichete:     ", etichete)


def arata(imagine):
    """Tensorul normalizat, adus înapoi la o imagine care se poate afișa."""
    x = imagine.permute(1, 2, 0).numpy() * np.array(ABATERE) + np.array(MEDIE)
    return np.clip(x, 0, 1)


fig, axe = plt.subplots(2, 4, figsize=(10, 5.4))
for ax, img, et in zip(axe.flat, imagini, etichete):
    ax.imshow(arata(img))
    ax.set_title(f"{CLASE[et]} (eticheta {et.item()})", fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 2. Referința simplă

Pasul din flux: pregătirea evaluării.

Înainte de orice model, te întrebi: cât de bine ar ieși cineva care nu se uită deloc la imagine și răspunde mereu cu aceeași clasă? Aceasta este referința simplă. Un model care nu o depășește clar nu a învățat nimic util, oricât de complicat ar fi. La cursul 1, la prognoză, metoda simplă a avut o eroare mai mică decât rețeaua neuronală: de aceea comparăm mereu.

### Exercițiul B2 (completează)

Folosește numerele afișate în secțiunea 1. Ce acuratețe pe validare are cineva care răspunde mereu „bees”? În celula de mai jos, înlocuiește `None` cu calculul tău (de exemplu `12 / 34`) și rulează. Celula afișează apoi cele două variante ale referinței: clasa cea mai frecventă pe validare și clasa cea mai frecventă la antrenare.

In [ ]:
# Exercițiul B2: înlocuiește None cu calculul tău, de exemplu 12 / 34
estimarea_ta = None

numar_antrenare = Counter(date_antrenare.targets)
numar_validare = Counter(date_validare.targets)

# clasa cea mai frecventă pe validare: cea mai bună acuratețe pe care o poate avea un răspuns constant
clasa_val, cate_val = numar_validare.most_common(1)[0]
referinta_val = cate_val / len(date_validare)
# clasa cea mai frecventă la antrenare: ce ar alege cineva care a văzut doar datele de antrenare
clasa_antr = numar_antrenare.most_common(1)[0][0]
referinta_antr = numar_validare[clasa_antr] / len(date_validare)

print(f"Mereu {CLASE[clasa_val]} (cea mai frecventă pe validare):   {procent(referinta_val)}")
print(f"Mereu {CLASE[clasa_antr]} (cea mai frecventă la antrenare): {procent(referinta_antr)}")

REFERINTA = max(referinta_val, referinta_antr)  # comparăm cu varianta cea mai greu de depășit
print("Referința simplă folosită mai departe:", procent(REFERINTA))

if estimarea_ta is None:
    print("\nB2: nu ai completat încă estimarea_ta.")
else:
    print(f"\nB2: estimarea ta este {procent(estimarea_ta)}; răspunsul exact este {procent(referinta_val)}.")

Răspunsul tău: (o frază) De ce comparăm mai departe modelul cu valoarea mai mare dintre cele două referințe, nu cu cea mai mică?

## 3. Modelul preantrenat, fără antrenare

Pasul din flux: model.

ResNet-18 este o rețea convoluțională cu 18 straturi, antrenată pe ImageNet: peste un milion de fotografii, în 1.000 de categorii (câini, mașini, fructe, insecte…). PyTorch o descarcă împreună cu ponderile învățate (`weights=ResNet18_Weights.DEFAULT`). Întâi o folosim așa cum este, fără nicio antrenare: ce răspunde la imaginile noastre?

In [ ]:
from torchvision.models import ResNet18_Weights

PONDERI = ResNet18_Weights.DEFAULT
model_imagenet = models.resnet18(weights=PONDERI).to(DISPOZITIV).eval()
CATEGORII = PONDERI.meta["categories"]  # cele 1000 de nume din ImageNet, în engleză

print("Categorii ImageNet:", len(CATEGORII))
for nume in ["ant", "bee"]:
    print(f"{nume}: categoria {CATEGORII.index(nume)}" if nume in CATEGORII else f"{nume}: lipsește")
print("Primele cinci categorii:", CATEGORII[:5])

In [ ]:
@torch.no_grad()
def top3_imagenet(imagine):
    """Primele trei categorii ImageNet pentru o imagine, cu probabilitățile lor."""
    probabilitati = model_imagenet(imagine.unsqueeze(0).to(DISPOZITIV)).softmax(dim=1)[0]
    p, idx = probabilitati.topk(3)
    return [(CATEGORII[i], pi.item()) for pi, i in zip(p, idx)]


exemple = [0, 20, 40, 75, 100, 125]  # trei furnici și trei albine din validare
fig, axe = plt.subplots(1, 6, figsize=(15, 3.8))
for ax, i in zip(axe, exemple):
    imagine, eticheta = date_validare[i]
    ax.imshow(arata(imagine))
    ax.set_title(f"real: {CLASE[eticheta]}\n" + "\n".join(f"{n} {procent(p)}" for n, p in top3_imagenet(imagine)),
                 fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# același lucru pe toată validarea: cât de des răspunde exact „ant” la furnici și „bee” la albine?
CATEGORIA_IMAGENET = {"ants": "ant", "bees": "bee"}
raspunsuri = []
with torch.no_grad():
    for imagini, _ in incarcator_validare:
        raspunsuri += model_imagenet(imagini.to(DISPOZITIV)).argmax(dim=1).tolist()

exacte = [CATEGORII[r] == CATEGORIA_IMAGENET[CLASE[y]] for r, y in zip(raspunsuri, date_validare.targets)]
print("Răspunsul exact (ant la furnici, bee la albine):", procent(sum(exacte) / len(exacte)))
print("Cele mai frecvente răspunsuri:")
for nume, cate in Counter(CATEGORII[r] for r in raspunsuri).most_common(10):
    print(f"  {nume:20} {cate}")

Ce observi:

1. Furnica și albina sunt printre cele 1.000 de categorii, iar modelul le recunoaște de multe ori. Pentru o parte din imagini alege însă altceva (o altă insectă, o floare, chiar un obiect): răspunde dintr-o listă de 1.000 și nu știe că întrebarea noastră are doar două răspunsuri.
2. Ieșirea lui are 1.000 de numere, nu 2. Ca să răspundă la întrebarea noastră, ultimul strat trebuie înlocuit.
3. Pentru clasele unei probleme reale (piese cu defecte, soiuri de plante, tipuri de documente) nu există o categorie în ImageNet.

Ce păstrăm din model: straturile care au învățat să vadă margini, texturi și forme. Ce schimbăm: ultimul strat, cel care decide între clase. Aceasta este învățarea prin transfer.

## 4. Învățarea prin transfer

Pașii din flux: eroare și antrenare.

1. Încărcăm din nou ResNet-18 preantrenat și îi înghețăm toate straturile (`requires_grad = False`): ponderile lor nu se mai schimbă.
2. Înlocuim ultimul strat (`fc`: 512 intrări, 1.000 de ieșiri) cu unul nou, cu 512 intrări și 2 ieșiri, câte un scor pentru fiecare clasă.
3. Antrenăm doar stratul nou. Eroarea (pierderea) se măsoară cu entropia încrucișată (cross-entropy): este mică atunci când modelul dă o probabilitate mare clasei corecte. Optimizatorul Adam schimbă ponderile stratului nou, pas cu pas, ca eroarea să scadă.

O epocă înseamnă o trecere prin toate cele 244 de imagini de antrenare (16 loturi). După fiecare epocă afișăm pierderea și acuratețea pe antrenare și pe validare.

In [ ]:
# Parametrii antrenării (Exercițiul B3: îi schimbi aici)
NR_EPOCI = 5            # câte treceri prin datele de antrenare
RATA_INVATARE = 0.001   # cât de mare este fiecare pas al optimizatorului

In [ ]:
def model_transfer(nr_clase):
    """ResNet-18 preantrenat, cu straturile înghețate și un ultim strat nou, pentru nr_clase clase."""
    model = models.resnet18(weights=PONDERI)
    for parametru in model.parameters():
        parametru.requires_grad = False                     # pasul 1: înghețăm
    model.fc = nn.Linear(model.fc.in_features, nr_clase)    # pasul 2: stratul nou, antrenabil
    return model.to(DISPOZITIV)


def numar(n):
    """11689512 -> '11.689.512'"""
    return f"{n:,}".replace(",", ".")


torch.manual_seed(SAMANTA)
model = model_transfer(len(CLASE))
total = sum(p.numel() for p in model.parameters())
antrenabili = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Parametri în total:   ", numar(total))
print("Parametri antrenabili:", numar(antrenabili), f"({100 * antrenabili / total:.3f}% din total)".replace(".", ","))

Din cele aproape 11,2 milioane de parametri antrenăm doar 1.026: 512 × 2 ponderi și 2 termeni liberi (bias), câte unul pentru fiecare clasă.

În funcția `o_epoca` de mai jos apare `model.eval()` și la antrenare. De obicei se scrie `model.train()`; aici straturile sunt înghețate, iar modul eval le păstrează neschimbate și statisticile de normalizare (BatchNorm). Revii la acest detaliu la rețelele convoluționale.

In [ ]:
criteriu = nn.CrossEntropyLoss()  # eroarea


def o_epoca(model, incarcator, optimizator=None):
    """O trecere prin date. Cu optimizator: antrenare; fără optimizator: doar evaluare."""
    model.eval()  # straturile înghețate rămân exact cum au fost învățate
    suma_pierderi, corecte, n = 0.0, 0, 0
    for imagini, etichete in incarcator:
        imagini, etichete = imagini.to(DISPOZITIV), etichete.to(DISPOZITIV)
        with torch.set_grad_enabled(optimizator is not None):
            iesiri = model(imagini)                  # modelul: câte un scor pentru fiecare clasă
            pierdere = criteriu(iesiri, etichete)    # eroarea pe acest lot
        if optimizator is not None:
            optimizator.zero_grad()
            pierdere.backward()                      # gradientul, doar pentru parametrii antrenabili
            optimizator.step()                       # antrenarea: un pas care micșorează eroarea
        suma_pierderi += pierdere.item() * len(etichete)
        corecte += (iesiri.argmax(dim=1) == etichete).sum().item()
        n += len(etichete)
    return suma_pierderi / n, corecte / n


def antreneaza(model, incarcator_antr, incarcator_val, nr_epoci, rata, parametri=None):
    """Antrenează și întoarce istoricul: (pierdere, acuratețe) pe antrenare și validare, după fiecare epocă."""
    if parametri is None:
        parametri = [p for p in model.parameters() if p.requires_grad]
    optimizator = torch.optim.Adam(parametri, lr=rata)
    istoric = []
    for epoca in range(1, nr_epoci + 1):
        t = time.time()
        p_antr, a_antr = o_epoca(model, incarcator_antr, optimizator)
        p_val, a_val = o_epoca(model, incarcator_val)
        istoric.append((p_antr, a_antr, p_val, a_val))
        print(f"epoca {epoca}/{nr_epoci}  pierdere {p_antr:.3f} / {p_val:.3f}  "
              f"acuratețe {procent(a_antr)} / {procent(a_val)}  (antrenare / validare; {time.time() - t:.0f} s)")
    return istoric


istoric = antreneaza(model, incarcator_antrenare, incarcator_validare, NR_EPOCI, RATA_INVATARE)

In [ ]:
def grafic_istoric(istoric, titlu=""):
    """Pierderea și acuratețea după fiecare epocă; linia punctată este referința simplă."""
    epoci = list(range(1, len(istoric) + 1))
    fig, axe = plt.subplots(1, 2, figsize=(11, 3.8))
    for ax, k, nume in [(axe[0], 0, "pierderea"), (axe[1], 1, "acuratețea")]:
        antr, val = [h[k] for h in istoric], [h[k + 2] for h in istoric]
        ax.plot(epoci, antr, color=TEXT, lw=2, marker="o")
        ax.plot(epoci, val, color=ACCENT, lw=2.5, marker="o")
        ax.text(epoci[-1] + 0.15, antr[-1], "antrenare", color=TEXT, va="center")
        ax.text(epoci[-1] + 0.15, val[-1], "validare", color=ACCENT, va="center")
        if k == 1:
            ax.axhline(REFERINTA, color=SECUNDAR, ls="--", lw=1)
            ax.text(1, REFERINTA + 0.01, "referința simplă", color=SECUNDAR, va="bottom")
        ax.set_title(nume + titlu, loc="left")
        ax.set_xlabel("epoca")
        ax.set_xticks(epoci)
        ax.set_xlim(0.7, epoci[-1] + 1.4)
    plt.tight_layout()
    plt.show()


grafic_istoric(istoric)

### Exercițiul B3 (modifică și notează)

Schimbă în celula cu parametri, pe rând, `NR_EPOCI` (1, apoi 10) și `RATA_INVATARE` (`0.0001`, apoi `0.1`), lăsând celălalt parametru la valoarea inițială. După fiecare schimbare rulezi din nou, cu Shift+Enter, cele patru celule de la parametri până la grafic și notezi în tabel acuratețea și pierderea pe validare din ultima epocă. La final pui înapoi valorile inițiale (5 și `0.001`) și rulezi din nou aceleași celule, ca restul caietului să folosească modelul de bază.

Răspunsul tău:

| Varianta | Acuratețea pe validare | Pierderea pe validare |
|---|---|---|
| 5 epoci, rata `0.001` (inițial) | | |
| 1 epocă, rata `0.001` | | |
| 10 epoci, rata `0.001` | | |
| 5 epoci, rata `0.0001` | | |
| 5 epoci, rata `0.1` | | |

Ce ai observat (2–3 fraze; uită-te și la pierdere, nu doar la acuratețe):

## 5. Evaluarea

Pasul din flux: evaluare.

Comparăm modelul antrenat cu referința simplă, apoi ne uităm unde greșește. Matricea de confuzie numără, pentru fiecare clasă reală, ce a prezis modelul; galeria arată chiar imaginile greșite.

In [ ]:
@torch.no_grad()
def predictii(model, incarcator):
    """Etichetele reale, clasele prezise și probabilitatea clasei prezise, pentru tot setul."""
    model.eval()
    reale, prezise, prob = [], [], []
    for imagini, etichete in incarcator:
        p = model(imagini.to(DISPOZITIV)).softmax(dim=1).cpu()
        prob_max, clasa = p.max(dim=1)
        reale += etichete.tolist()
        prezise += clasa.tolist()
        prob += prob_max.tolist()
    return np.array(reale), np.array(prezise), np.array(prob)


reale, prezise, prob = predictii(model, incarcator_validare)
ACURATETE = (reale == prezise).mean()
print("Acuratețea modelului pe validare:", procent(ACURATETE))
print("Referința simplă:                ", procent(REFERINTA))
print("Diferența:                       ", f"{100 * (ACURATETE - REFERINTA):.1f}".replace(".", ","), "puncte procentuale")

In [ ]:
def matrice_confuzie(reale, prezise, nr_clase):
    """m[i, j] = câte imagini din clasa reală i au fost prezise ca j."""
    m = np.zeros((nr_clase, nr_clase), dtype=int)
    for r, p in zip(reale, prezise):
        m[r, p] += 1
    return m


def arata_confuzia(m, clase):
    fig, ax = plt.subplots(figsize=(1.6 + 1.3 * len(clase), 1.2 + 1.3 * len(clase)))
    ax.imshow(m, cmap=LinearSegmentedColormap.from_list("cursul", ["#FFFFFF", SERIE2]))
    for i in range(len(clase)):
        for j in range(len(clase)):
            ax.text(j, i, m[i, j], ha="center", va="center", fontsize=14,
                    color="#FFFFFF" if m[i, j] > m.max() / 2 else TEXT)
    ax.set_xticks(range(len(clase)), clase)
    ax.set_yticks(range(len(clase)), clase)
    ax.set_xlabel("clasa prezisă")
    ax.set_ylabel("clasa reală")
    for s in ax.spines.values():
        s.set_visible(False)
    plt.tight_layout()
    plt.show()


M = matrice_confuzie(reale, prezise, len(CLASE))
print("Rândurile: clasa reală; coloanele: clasa prezisă")
for i, c in enumerate(CLASE):
    print(f"  {c}: {M[i].tolist()}")
arata_confuzia(M, CLASE)

In [ ]:
def galerie_greseli(date, reale, prezise, prob, clase, maxim=8):
    """Imaginile greșit clasificate (cel mult maxim), cu clasa reală, clasa prezisă și probabilitatea."""
    gresite = np.where(reale != prezise)[0]
    print(f"Imagini greșit clasificate: {len(gresite)} din {len(reale)}")
    if len(gresite) == 0:
        return
    alese = gresite[:maxim]
    coloane = min(4, len(alese))
    randuri = (len(alese) + coloane - 1) // coloane
    fig, axe = plt.subplots(randuri, coloane, figsize=(3 * coloane, 3.6 * randuri), squeeze=False)
    for ax in axe.flat:
        ax.axis("off")
    for ax, i in zip(axe.flat, alese):
        ax.imshow(arata(date[i][0]))
        ax.set_title(f"nr. {i}: real {clase[reale[i]]}\nprezis {clase[prezise[i]]} ({procent(prob[i])})", fontsize=10)
    plt.tight_layout()
    plt.show()


galerie_greseli(date_validare, reale, prezise, prob, CLASE)

## 6. Modelul pe o imagine nouă

Pasul din flux: folosirea modelului.

Un model antrenat se folosește pe imagini pe care nu le-a văzut. Funcția `clasifica` primește o cale de fișier sau un URL, pregătește imaginea exact ca la antrenare și întoarce clasa prezisă și probabilitatea ei.

In [ ]:
def deschide_imagine(sursa):
    """Deschide o imagine dintr-un fișier sau de la un URL."""
    if str(sursa).startswith(("http://", "https://")):
        cerere = urllib.request.Request(sursa, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(cerere, timeout=30) as raspuns:
            sursa = io.BytesIO(raspuns.read())
    imagine = Image.open(sursa)
    return ImageOps.exif_transpose(imagine).convert("RGB")  # rotația din telefon, apoi culorile RGB


@torch.no_grad()
def clasifica(sursa, model_folosit=None, clase=None, afiseaza=True):
    """Întoarce (clasa, probabilitatea) pentru o imagine dată prin cale sau URL."""
    if model_folosit is None:
        model_folosit = model
    if clase is None:
        clase = CLASE
    imagine = deschide_imagine(sursa)
    x = pregatire(imagine).unsqueeze(0).to(DISPOZITIV)  # un lot cu o singură imagine
    model_folosit.eval()
    probabilitati = model_folosit(x).softmax(dim=1)[0].cpu()
    k = probabilitati.argmax().item()
    if afiseaza:
        plt.figure(figsize=(3.2, 3.2))
        plt.imshow(imagine)
        plt.axis("off")
        plt.title(f"{clase[k]} ({procent(probabilitati[k].item())})")
        plt.show()
    return clase[k], round(probabilitati[k].item(), 4)


# o imagine din validare
cale, eticheta = date_validare.samples[100]
print(cale, "| clasa reală:", CLASE[eticheta])
print(clasifica(cale))

In [ ]:
# o imagine care nu e nici furnică, nici albină (un câine), luată de pe internet
URL_CAINE = "https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg"
try:
    print(clasifica(URL_CAINE))
except Exception as e:
    print("Imaginea nu a putut fi descărcată acum:", e)

### Poza ta

Ca să încerci o poză proprie (de pe telefon sau de pe calculator), pui `INCARCA_POZA = True` în celula de mai jos, o rulezi și alegi fișierul (JPG sau PNG; o fotografie HEIC de pe iPhone o transformi întâi în JPG). După ce ai văzut rezultatul, pui înapoi `False`: altfel, la „Run all”, caietul se oprește și așteaptă un fișier.

In [ ]:
INCARCA_POZA = False  # True: alegi o poză; înainte de predare, înapoi False

if INCARCA_POZA:
    from google.colab import files       # există doar în Colab
    for nume in files.upload():          # deschide fereastra de alegere a fișierului
        print(nume, clasifica(nume))

### Exercițiul B4 (interpretează)

Uită-te la galeria imaginilor greșite din secțiunea 5 și la rezultatul pentru câine.

1. Alege două imagini greșite și spune, pentru fiecare, ce crezi că l-a încurcat pe model (unghiul, mărimea insectei, fundalul, mai multe insecte în imagine…).
2. Ce clasă și ce probabilitate a dat modelul pentru câine? De ce răspunde așa, deși în imagine nu e nici furnică, nici albină? Ce ar trebui schimbat ca modelul să poată răspunde „altceva”?

Răspunsul tău:

## Extensii (opționale, pentru nota 8–10)

Alegi cel puțin una. Fiecare are un comutator: îl pui pe `True`, rulezi celula și scrii, în celula „Comentariul tău” de sub ea, ce ai schimbat, ce rezultat ai obținut față de modelul de bază și cum îl explici. Extensia pe care o predai rămâne cu comutatorul pe `True`, ca rezultatele ei să se vadă în caiet.

Rulează extensiile după ce nivelul de bază este gata. Pe procesor, E1 durează mai mult decât nivelul de bază; dacă vrei să meargă mai repede, alegi Runtime → Change runtime type → T4 GPU (când Colab are GPU disponibil) și rulezi din nou tot caietul.

### Extensia E1: reglajul fin (fine-tuning)

Până acum am antrenat doar ultimul strat. Acum dezghețăm și ultimul bloc al rețelei (`layer4`) și îl antrenăm cu o rată de învățare de 10 ori mai mică decât cea a stratului nou, ca să nu stricăm ce a învățat deja pe ImageNet. Compară acuratețea pe validare, numărul de parametri antrenabili și durata unei epoci cu modelul de bază.

In [ ]:
RULEAZA_E1 = False  # True: rulezi extensia

if RULEAZA_E1:
    torch.manual_seed(SAMANTA)
    model_e1 = model_transfer(len(CLASE))
    for parametru in model_e1.layer4.parameters():
        parametru.requires_grad = True                     # dezghețăm ultimul bloc
    print("Parametri antrenabili:", numar(sum(p.numel() for p in model_e1.parameters() if p.requires_grad)))
    grupuri = [{"params": model_e1.layer4.parameters(), "lr": RATA_INVATARE / 10},  # pași mici pentru blocul vechi
               {"params": model_e1.fc.parameters(), "lr": RATA_INVATARE}]           # pași obișnuiți pentru stratul nou
    istoric_e1 = antreneaza(model_e1, incarcator_antrenare, incarcator_validare, NR_EPOCI, RATA_INVATARE, grupuri)
    grafic_istoric(istoric_e1, " (E1, reglaj fin)")
    print("Validare: doar ultimul strat", procent(ACURATETE), "| reglaj fin", procent(istoric_e1[-1][3]))
else:
    print("E1 nu rulează (RULEAZA_E1 = False).")

Comentariul tău (E1):

### Extensia E2: augmentarea datelor

Augmentarea schimbă la întâmplare imaginile de antrenare la fiecare epocă (o altă tăietură, oglindire, altă lumină), ca modelul să vadă mai multe variante ale acelorași 244 de fotografii. Imaginile de validare rămân neschimbate. Compară cu modelul de bază. Indiciu: cu straturile înghețate și puține epoci, efectul poate fi mic; încearcă și cu mai multe epoci, și explică ce vezi.

In [ ]:
RULEAZA_E2 = False  # True: rulezi extensia

if RULEAZA_E2:
    augmentare = transforms.Compose([
        transforms.RandomResizedCrop(224, scale=(0.6, 1.0)),  # o tăietură aleasă la întâmplare
        transforms.RandomHorizontalFlip(),                     # oglindire stânga-dreapta
        transforms.ColorJitter(0.2, 0.2, 0.2),                 # lumină, contrast, saturație
        transforms.ToTensor(),
        transforms.Normalize(MEDIE, ABATERE),
    ])
    date_antrenare_aug = datasets.ImageFolder(os.path.join(DATE, "train"), augmentare)
    fig, axe = plt.subplots(1, 6, figsize=(14, 2.6))     # aceeași imagine, de șase ori, de fiecare dată altfel
    for ax in axe:
        ax.imshow(arata(date_antrenare_aug[0][0]))
        ax.axis("off")
    plt.show()
    incarcator_aug = torch.utils.data.DataLoader(
        date_antrenare_aug, batch_size=MARIME_LOT, shuffle=True, num_workers=NR_FIRE,
        generator=torch.Generator().manual_seed(SAMANTA))
    torch.manual_seed(SAMANTA)
    model_e2 = model_transfer(len(CLASE))
    istoric_e2 = antreneaza(model_e2, incarcator_aug, incarcator_validare, NR_EPOCI, RATA_INVATARE)
    grafic_istoric(istoric_e2, " (E2, cu augmentare)")
    print("Validare: fără augmentare", procent(ACURATETE), "| cu augmentare", procent(istoric_e2[-1][3]))
else:
    print("E2 nu rulează (RULEAZA_E2 = False).")

Comentariul tău (E2):

### Extensia E3: setul tău de imagini

Faci câte 15–20 de fotografii pentru 2–3 clase alese de tine (de exemplu căni, pahare și sticle; frunze de stejar și de arțar). Pui fotografiile fiecărei clase într-un folder cu numele clasei, toate în folderul `setul_meu`, apoi arhivezi folderul ca `setul_meu.zip`:

```
setul_meu/
    cani/      foto1.jpg, foto2.jpg, ...
    pahare/    ...
    sticle/    ...
```

În Colab, tragi `setul_meu.zip` în panoul Files din bara din stânga sau pui `INCARCA_ZIP = True` mai jos și alegi arhiva (numele trebuie să fie exact `setul_meu.zip`). Codul dezarhivează, împarte la întâmplare fiecare clasă în antrenare (75%) și validare (25%) și antrenează ultimul strat, ca în secțiunea 4. Nu folosești fotografii cu persoane, cu documente personale sau cu date interne de la locul de muncă. Celula nu rulează dacă setul lipsește.

In [ ]:
INCARCA_ZIP = False   # True: alegi setul_meu.zip de pe calculator (doar în Colab)
FOLDER_SET = "setul_meu"

if INCARCA_ZIP:
    from google.colab import files
    files.upload()
if not os.path.isdir(FOLDER_SET) and os.path.exists(FOLDER_SET + ".zip"):
    with zipfile.ZipFile(FOLDER_SET + ".zip") as z:
        z.extractall(FOLDER_SET)
    shutil.rmtree(os.path.join(FOLDER_SET, "__MACOSX"), ignore_errors=True)  # resturi din arhivele făcute pe Mac
    if os.path.isdir(os.path.join(FOLDER_SET, FOLDER_SET)):                   # arhiva conținea chiar folderul
        FOLDER_SET = os.path.join(FOLDER_SET, FOLDER_SET)


def citeste(cale):
    return ImageOps.exif_transpose(Image.open(cale)).convert("RGB")  # rotația din telefon


def e_imagine(cale):
    nume = os.path.basename(cale)
    return nume.lower().endswith((".jpg", ".jpeg", ".png")) and not nume.startswith(".")


if os.path.isdir(FOLDER_SET):
    setul_meu = datasets.ImageFolder(FOLDER_SET, pregatire, loader=citeste, is_valid_file=e_imagine)
    clase_mele = setul_meu.classes
    print("Clase:", clase_mele, "| imagini pe clasă:", dict(Counter(clase_mele[t] for t in setul_meu.targets)))
    g = torch.Generator().manual_seed(SAMANTA)
    idx_antr, idx_val = [], []
    for c in range(len(clase_mele)):                       # împărțirea, separat pe fiecare clasă
        idx = [i for i, t in enumerate(setul_meu.targets) if t == c]
        idx = [idx[j] for j in torch.randperm(len(idx), generator=g).tolist()]
        k = max(1, round(0.25 * len(idx)))
        idx_val += idx[:k]
        idx_antr += idx[k:]
    antr_meu = torch.utils.data.Subset(setul_meu, idx_antr)
    val_meu = torch.utils.data.Subset(setul_meu, idx_val)
    inc_antr_meu = torch.utils.data.DataLoader(antr_meu, batch_size=MARIME_LOT, shuffle=True,
                                               num_workers=NR_FIRE, generator=g)
    inc_val_meu = torch.utils.data.DataLoader(val_meu, batch_size=MARIME_LOT, num_workers=NR_FIRE)
    referinta_mea = Counter(setul_meu.targets[i] for i in idx_val).most_common(1)[0][1] / len(idx_val)
    print("Antrenare:", len(idx_antr), "| validare:", len(idx_val), "| referința simplă:", procent(referinta_mea))
    torch.manual_seed(SAMANTA)
    model_e3 = model_transfer(len(clase_mele))
    istoric_e3 = antreneaza(model_e3, inc_antr_meu, inc_val_meu, NR_EPOCI, RATA_INVATARE)
    r3, p3, pr3 = predictii(model_e3, inc_val_meu)
    print("Acuratețea pe validare:", procent((r3 == p3).mean()), "| referința simplă:", procent(referinta_mea))
    arata_confuzia(matrice_confuzie(r3, p3, len(clase_mele)), clase_mele)
    galerie_greseli(val_meu, r3, p3, pr3, clase_mele)
else:
    print(f"E3 nu rulează: nu există folderul {FOLDER_SET} și nici arhiva {FOLDER_SET}.zip.")

Comentariul tău (E3): ce clase ai ales, cum ai făcut fotografiile, ce acuratețe ai obținut față de referința simplă și ce imagini au fost greșite.

### Extensia E4: alt model preantrenat

MobileNetV3-Small este o rețea făcută pentru telefoane: are de câteva ori mai puțini parametri decât ResNet-18 și rulează mai repede. Îi înlocuim ultimul strat și îl antrenăm exact ca în secțiunea 4. Compară acuratețea pe validare, numărul de parametri și durata unei epoci cu ResNet-18.

In [ ]:
RULEAZA_E4 = False  # True: rulezi extensia

if RULEAZA_E4:
    from torchvision.models import MobileNet_V3_Small_Weights
    torch.manual_seed(SAMANTA)
    model_e4 = models.mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.DEFAULT)
    for parametru in model_e4.parameters():
        parametru.requires_grad = False                                         # înghețăm
    model_e4.classifier[3] = nn.Linear(model_e4.classifier[3].in_features, len(CLASE))  # ultimul strat, nou
    model_e4 = model_e4.to(DISPOZITIV)
    print("Parametri în total:", numar(sum(p.numel() for p in model_e4.parameters())),
          "| antrenabili:", numar(sum(p.numel() for p in model_e4.parameters() if p.requires_grad)))
    istoric_e4 = antreneaza(model_e4, incarcator_antrenare, incarcator_validare, NR_EPOCI, RATA_INVATARE)
    grafic_istoric(istoric_e4, " (E4, MobileNetV3-Small)")
    print("Validare: ResNet-18", procent(ACURATETE), "| MobileNetV3-Small", procent(istoric_e4[-1][3]))
else:
    print("E4 nu rulează (RULEAZA_E4 = False).")

Comentariul tău (E4):

## Verificarea finală

Celula de mai jos are sens după Runtime → Restart session and run all: dacă afișează mesajul, caietul rulează de la început până la sfârșit fără erori.

In [ ]:
durata = (time.time() - T_START) / 60
print(f"Caietul a rulat până la capăt în {durata:.1f} minute, pe {DISPOZITIV.type.upper()}.".replace(".", ",", 1))
print("Modelul de bază pe validare:", procent(ACURATETE), "| referința simplă:", procent(REFERINTA))

## Ce ai învățat

- Un model preantrenat pe ImageNet se adaptează la o problemă nouă cu câteva sute de imagini, antrenând doar ultimul strat: o mie de parametri din peste 11 milioane.
- Ai parcurs, pas cu pas, fluxul de la curs, de la date la evaluare: date, model, eroare, antrenare, evaluare.
- Acuratețea are sens doar lângă o referință simplă; matricea de confuzie arată unde greșește modelul, iar imaginile greșite te ajută să înțelegi de ce.
- Modelul răspunde mereu cu una dintre clasele învățate, chiar și pentru o imagine care nu aparține niciuneia.

## Legătura cu cursul 3

Lotul de imagini a fost un tensor cu forma `[16, 3, 224, 224]`, iar `pierdere.backward()` a calculat gradientul erorii doar pentru stratul nou, pentru că celelalte aveau `requires_grad = False`. La cursul 3 vezi ce sunt tensorii și gradientul și urmărești în cod, pas cu pas, ce a făcut aici optimizatorul, pe regresia liniară și pe cea logistică.

## Declarație AI

Completezi la fiecare caiet predat, la persoana întâi:

- Instrumentul AI: …
- Pentru ce l-am folosit: …
- Ce am verificat singur (și cum): …

Dacă nu ai folosit niciun instrument AI, înlocuiești cele trei rânduri cu: Nu am folosit asistenți AI.